In [1]:
import os
import json
import numpy as np
import pandas as pd

In [2]:
# ANALYSEPARAMETER - Reaktionsschwelle & Reaktionsfenster

In [ ]:
def formel(df):
    lambdaFactor = 2

    # 1. Renditen
    returns = df['Close'].pct_change().dropna()
        
    # 2. Standardabweichung
    sigma = returns.std()
    
    # 3. Standardisierung
    z = returns / sigma
    absZ = z.abs()
    
    # Reaktionsschwelle - 5%
    tau = absZ.quantile(0.95)
    
    # Reaktionsfenster
    m = absZ.mean()
    W = lambdaFactor * (tau / m)
    WStar = int(np.ceil(W))

    return tau, WStar, sigma

In [ ]:
# GLOBAL Berechnung Analyseparameter

In [ ]:
def loadAssetsDataframe(folderName = "datenCLEAN"):
    assets = {}
    for filename in os.listdir(folderName):
        assetName = filename.split("_")[0]
        df = pd.read_csv(f"{folderName}/{filename}", parse_dates=["Date"])
        assets[assetName] = df
    return assets

def calculateGlobalAnalysisParameter(assetsDataframe):
    allTau = {}     
    allWStar = {}   
    allSigma = {}

    for asset, df in assetsDataframe.items():

        tau, WStar, sigma = formel(df)

        allTau[asset] = tau
        allWStar[asset] = WStar
        allSigma[asset] = sigma

    return allTau, allWStar, allSigma

assetsDataframe = loadAssetsDataframe(folderName = "datenCLEAN")
allTau, allWStar, allSigma = calculateGlobalAnalysisParameter(assetsDataframe)

# with open("analyseparameter/reaktionsschwelle.json", "w") as f:
#     json.dump(allTau, f)

# with open("analyseparameter/reaktionsfenster.json", "w") as f:
#     json.dump(allWStar, f)

# with open("analyseparameter/standardabweichung.json", "w") as f:
#     json.dump(allSigma, f)

In [ ]:
# PRO JAHR Berechnung Analyseparameter

In [ ]:
def createYearSeperation(folderName):
    ALLAssetsYearSeperated = {}
    for filename in os.listdir(folderName):
        assetYearSeperated = {}
        assetName = filename.split("_")[0]
        df = pd.read_csv(f"{folderName}/{filename}", parse_dates=["Date"], index_col="Date")
        years = df.index.year.unique()
        for year in years:
            yearDF = df[df.index.year == year]
            assetYearSeperated[year] = yearDF
        ALLAssetsYearSeperated[assetName] = assetYearSeperated
    return ALLAssetsYearSeperated

def calculateAnalysisParametersPERYEAR(assetYearData):
    allParameterPerAssetPerYear = {}
    for asset, yearsDict in assetYearData.items():
        allParameterPerAssetPerYear[asset] = {}
        for year, df in yearsDict.items():
            tau, WStar, sigma = formel(df)
            allParameterPerAssetPerYear[asset][year] = {
                "tau": tau,
                "window": WStar,
                "sigma": sigma
            }
    return allParameterPerAssetPerYear

assetsDataframePerYear = createYearSeperation(folderName = "datenCLEAN")
allParameterPerAssetPerYear = calculateAnalysisParametersPERYEAR(assetsDataframePerYear)

# with open("analyseparameter/analysisParameterPERYEAR.json", "w") as f:
#     json.dump(allParameterPerAssetPerYear, f)